# Lihua AI - Train Qwen2.5-Coder-7B (Unsloth + QLoRA)

**Before you start**
1. Upload `lihua_train.csv` to the main folder of your Google Drive ("My Drive").
2. Menu: Runtime > Change runtime type > choose **T4 GPU** > Save.
3. Run the cells from top to bottom (click the play button on each cell, wait until it finishes).

In [ ]:
%%capture
!pip install unsloth

In [ ]:
# Cell 2: connect Google Drive (click Allow when asked)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: load the base model (about 5 GB download)
from unsloth import FastLanguageModel, is_bfloat16_supported

MAX_LEN = 1536
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    max_seq_length=MAX_LEN,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
# Cell 4: load your CSV and turn it into chat format
import pandas as pd
from datasets import Dataset

SYSTEM = "You are Lihua AI, a friendly programming assistant. Answer clearly, put code in fenced blocks with the language name, and explain briefly."

df = pd.read_csv("/content/drive/MyDrive/lihua_train.csv")
df = df.dropna(subset=["instruction", "response"])
df = df.sample(n=min(5000, len(df)), random_state=1)   # 5000 rows = about 1.5 to 2.5 hours on free T4

def to_text(row):
    msgs = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": str(row["instruction"])},
        {"role": "assistant", "content": str(row["response"])},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False)

df["text"] = df.apply(to_text, axis=1)
dataset = Dataset.from_pandas(df[["text"]], preserve_index=False)
print(len(dataset), "training examples")
print(dataset[0]["text"][:500])

In [ ]:
# Cell 5: TRAIN (this is the long step - keep this browser tab open)
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_LEN,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=500,            # 500 steps x 8 = 4000 examples. Lower = faster, higher = better
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
    ),
)
trainer.train()

In [ ]:
# Cell 6: quick test
FastLanguageModel.for_inference(model)
msgs = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "Write C++ code that prints hello world 20 times."},
]
inputs = tokenizer.apply_chat_template(
    msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
).to("cuda")
out = model.generate(**inputs, max_new_tokens=300, temperature=0.3, do_sample=True)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

In [ ]:
# Cell 7: save a backup to Drive, then export the GGUF file for Ollama (takes about 10-15 minutes)
model.save_pretrained("/content/drive/MyDrive/lihua_lora")
tokenizer.save_pretrained("/content/drive/MyDrive/lihua_lora")

model.save_pretrained_gguf("lihua_gguf", tokenizer, quantization_method="q4_k_m")

import glob, shutil
files = glob.glob("/content/lihua_gguf*/**/*.gguf", recursive=True) + glob.glob("/content/*.gguf")
print("Found:", files)
shutil.copy(files[0], "/content/drive/MyDrive/lihua-7b.gguf")
print("DONE. Download lihua-7b.gguf from your Google Drive.")